# 01. Understanding the inputs to Stage 1

This notebook explains the real inputs that Rustamps Stage 1 needs.

It is the dataset-contract deep dive in the notebook set. Read `start_here.ipynb` first if you want the full repo map, then use this notebook when you want to understand why Stage 1 can or cannot run.

The goal is simple: before you run Stage 1, you should be able to answer three questions without reading source code:

- What arrays does Stage 1 need?
- Where do those arrays usually come from in a StaMPS-style dataset?
- Does my dataset look internally consistent enough for Stage 1 to succeed?

## Stage 1 in plain language

Stage 1 is the ingestion and organization step. It takes raw persistent-scatterer candidate data and turns it into structured outputs that the rest of the pipeline can use safely.

You do not need SAR expertise to follow this notebook. A practical mental model is enough:

- a **candidate** is one point that may behave like a stable radar scatterer
- a **patch** is one spatial tile of the dataset
- an **interferogram** is one phase-difference product between radar acquisitions
- Stage 1 reads raw candidate tables and phase values, checks that they agree, then writes organized outputs such as `ps1.mat`, `ph1.mat`, and `bp1.mat`

## The key idea: arrays are the contract, files are only one carrier

For users, this is the most important design idea in Rustamps:

- the pipeline really needs **arrays and scalars**
- text files, MAT files, NPY files, HDF5 files, pickle files, or in-memory objects are only **storage/transport formats**
- if the values, shapes, ordering, and dtypes are correct, the carrier format is secondary

So this notebook separates two views:

1. the **logical Stage-1 contract**: the arrays the algorithm needs
2. the **observed dataset inventory**: the concrete files found in the bundled example dataset

That separation is intentional. It makes the package easier to understand and harder to misuse.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
root = next(p for p in (here, *here.parents) if (p / "Cargo.toml").exists())
sys.path[:0] = [str(root), str(root / "notebooks")]

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from pystamps.input_contracts import describe_stage1_snap2stamps_flow, describe_stage_inputs
from pystamps.notebooks import inspect_stage1_inputs

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(suppress=True, precision=4)
pd.set_option('display.max_colwidth', 120)

REPO_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'Cargo.toml').exists() and (p / 'pystamps').exists())
DATASET = REPO_ROOT / 'inputs_and_outputs' / 'InSAR_dataset_test'
PATCH_NAME = 'PATCH_1'
DATASET_LABEL = DATASET.name

STAGE1_CONTRACT = describe_stage_inputs(1)[0]
ALL_STAGE_CONTRACTS = describe_stage_inputs('all')
SUMMARY = inspect_stage1_inputs(DATASET, patch_name=PATCH_NAME)

print(f'Dataset: {DATASET_LABEL}')
print(f'Patch:   {PATCH_NAME}')
print(f"Metadata mode: {SUMMARY['metadata_mode']}")
print(f"Warnings raised by package checks: {len(SUMMARY['warnings'])}")

## Logical Stage-1 contract

The next table is the cleanest definition of what Stage 1 needs.

Read it as an API contract:

- **array_name** is the logical object the algorithm expects
- **shape_hint** is the expected dimensionality
- **meaning** is the semantic role of that object
- **common_sources** gives typical storage formats, but those are examples, not the contract itself

If you have these arrays in memory with the correct meaning and shape, Stage 1 is conceptually ready to run.

In [ ]:
contract_df = pd.DataFrame(STAGE1_CONTRACT['inputs'])
display(contract_df)

## Whole-pipeline view

This notebook focuses on Stage 1, but the CLI can describe all pipeline stages.

The compact table below is useful when you want to see how the pipeline progresses from raw candidate ingestion to final filtered outputs.

In [ ]:
all_stages_df = pd.DataFrame([
    {'stage': item['stage'], 'title': item['title']}
    for item in ALL_STAGE_CONTRACTS
])
display(all_stages_df)

## Where these Stage-1 inputs come from in the SNAP to StaMPS workflow

Stage 1 does not create its raw inputs from the original SAR scenes. Those files are produced upstream by the SNAP-to-StaMPS preprocessing workflow.

Upstream context:

- the StaMPS project lists SNAP as one of the supported preprocessors
- the SNAP2StaMPS repository provides the concrete SNAP-side step sequence
- in that workflow, the final export step writes the StaMPS-compatible dataset that Rustamps Stage 1 reads

For Sentinel-1 TOPSAR, the SNAP2StaMPS README describes a step-by-step chain ending with `topsar_step_5_stamps_export_multiIW.py`. That export stage is the practical handoff point between SNAP products and the StaMPS-style patch files used here.

So the correct mental model is:

1. SNAP2StaMPS prepares and exports a StaMPS-compatible dataset
2. Rustamps Stage 1 ingests that exported dataset
3. Rustamps writes `ps1.mat`, `ph1.mat`, `bp1.mat`, and related MATLAB files for Stage 2

Primary upstream references:

- StaMPS repository: https://github.com/dbekaert/StaMPS
- SNAP2StaMPS repository: https://github.com/mdelgadoblasco/snap2stamps

In [ ]:
snap2stamps_flow_df = pd.DataFrame(describe_stage1_snap2stamps_flow())
display(snap2stamps_flow_df)

## What the package checks automatically

The next tables are generated by package-side inspection code, not handwritten notebook logic.

This is important because users should not have to infer data consistency by eye. The package should check the obvious failure modes early and say so clearly.

The consistency table answers questions such as:

- do the candidate tables have the same number of rows?
- does the phase stack have the same number of interferograms as the time and baseline metadata?
- are some inputs missing but recoverable at runtime?

The warning table summarizes concrete issues or limitations.

In [ ]:
consistency_df = pd.DataFrame(SUMMARY['consistency_rows'])
warnings_df = pd.DataFrame({'warning': SUMMARY['warnings']})

display(consistency_df)
if warnings_df.empty:
    display(Markdown('**No package warnings were raised for this Stage-1 inspection.**'))
else:
    display(warnings_df)

## Observed input inventory in the bundled dataset

Now we move from the logical contract to the concrete dataset.

The first table is a short summary of the inspected patch.
The second table lists the actual Stage-1 inputs that the package found or can derive at runtime.

This distinction matters:

- the contract says what Stage 1 needs
- the inventory says how this specific dataset supplies it

In [ ]:
overview_df = pd.DataFrame(SUMMARY['overview_rows'])
input_df = pd.DataFrame(SUMMARY['input_rows'])

display(overview_df)
display(input_df)

## Candidate-level preview

Stage 1 operates candidate by candidate. The preview below is deliberately small and readable.

It lets you connect the abstract contract to concrete rows:

- `candidate_id`, `azimuth_row`, and `range_col` come from the candidate index table
- `lon` and `lat` come from the geographic coordinate table
- `D_A` is an optional stability metric often used for quality control
- `hgt` appears only if the dataset carries an optional height prior

In [ ]:
candidate_preview_df = pd.DataFrame(SUMMARY['preview_rows'])
display(candidate_preview_df)

## Acquisition and interferogram metadata

Stage 1 also needs temporal context. Two levels are relevant:

- **acquisitions**: the image dates available in the stack
- **interferograms**: the master/slave date pairs whose phase differences were already formed

If the explicit text metadata files are missing, Rustamps can reconstruct the required arrays from `diff0` and `rslc` metadata. This is why the package reports a metadata mode rather than only present/missing files.

In [ ]:
acquisition_df = pd.DataFrame(SUMMARY['acquisition_rows'])
interferogram_df = pd.DataFrame(SUMMARY['interferogram_rows'])

display(acquisition_df.head(12))
display(interferogram_df.head(12))

## How Stage 1 prepares the MATLAB files

The raw files are not passed downstream directly. Stage 1 reorganizes them into a small set of MATLAB files that later stages consume.

This is the preparation pipeline inside Stage 1:

- raw candidate tables and phase values are loaded
- timing and baseline metadata are resolved or derived
- the phase stack is reordered in time and the master image is inserted
- candidate-linked arrays are sorted into one common order
- structured MATLAB outputs are written for the next stages

The two tables below show that preparation process explicitly.

In [ ]:
preparation_df = pd.DataFrame(SUMMARY['preparation_rows'])
mat_output_df = pd.DataFrame(SUMMARY['mat_output_rows'])

display(preparation_df)
display(mat_output_df)

## Plots: geometry, quality, and phase content

These plots answer the fastest sanity-check questions before a run:

1. Are the candidate points in plausible spatial locations?
2. Does the optional `D_A` metric look populated and well behaved?
3. Do the patch coordinates cover the expected image footprint?
4. Does the complex phase stack contain structured values rather than empty or obviously corrupted data?

In [ ]:
patch = SUMMARY['patch']
phase_preview = SUMMARY['phase_preview']
lonlat = patch['lonlat']
da = np.asarray(patch['da'], dtype=float).reshape(-1)
ij = patch['ij']

fig, axes = plt.subplots(2, 2, figsize=(16, 11), constrained_layout=True)
fig.suptitle('Stage-1 input checks for one patch', fontsize=15, fontweight='bold')

finite_da = np.isfinite(da)
if finite_da.any():
    scatter = axes[0, 0].scatter(lonlat[:, 0], lonlat[:, 1], c=da, s=2, cmap='viridis', alpha=0.5, linewidths=0)
    fig.colorbar(scatter, ax=axes[0, 0], fraction=0.046, pad=0.04, label='D_A')
else:
    axes[0, 0].scatter(lonlat[:, 0], lonlat[:, 1], s=2, color='tab:blue', alpha=0.5, linewidths=0)
axes[0, 0].set_title('Candidate footprint')
axes[0, 0].set_xlabel('longitude')
axes[0, 0].set_ylabel('latitude')

if finite_da.any():
    axes[0, 1].hist(da[finite_da], bins=50, color='tab:green', alpha=0.8)
    axes[0, 1].set_xlabel('D_A')
    axes[0, 1].set_ylabel('candidate count')
else:
    axes[0, 1].text(0.5, 0.5, 'No D_A values available in this patch', ha='center', va='center', transform=axes[0, 1].transAxes)
    axes[0, 1].set_axis_off()
axes[0, 1].set_title('D_A distribution')

axes[1, 0].scatter(ij[:, 2], ij[:, 1], s=2, c='tab:blue', alpha=0.35, linewidths=0)
axes[1, 0].set_title('Candidate positions in patch coordinates')
axes[1, 0].set_xlabel('range column')
axes[1, 0].set_ylabel('azimuth row')
axes[1, 0].invert_yaxis()

im = axes[1, 1].imshow(phase_preview['angle'], aspect='auto', cmap='twilight', interpolation='nearest')
axes[1, 1].set_title('Phase-angle preview from pscands.1.ph')
axes[1, 1].set_xlabel('sample interferogram index')
axes[1, 1].set_ylabel('sample candidate index')
fig.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04, label='phase angle [rad]')

plt.show()

## Optional metadata plots

These are secondary checks, but they are still useful:

- a height histogram tells you whether an optional height prior exists and looks plausible
- an acquisition timeline gives a quick view of temporal coverage

In [ ]:
height_values = np.asarray(SUMMARY['height_values'], dtype=float).reshape(-1)
acquisition_plot_df = pd.DataFrame(SUMMARY['acquisition_rows'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)

if height_values.size:
    axes[0].hist(height_values, bins=40, color='tab:orange', alpha=0.8)
    axes[0].set_xlabel('height')
    axes[0].set_ylabel('candidate count')
else:
    axes[0].text(0.5, 0.5, 'No pscands.1.hgt values are available for this patch', ha='center', va='center', transform=axes[0].transAxes)
    axes[0].set_axis_off()
axes[0].set_title('Height prior distribution')

if not acquisition_plot_df.empty:
    acquisition_plot_df = acquisition_plot_df.copy()
    acquisition_plot_df['date'] = pd.to_datetime(acquisition_plot_df['date'])
    axes[1].plot(acquisition_plot_df['date'], acquisition_plot_df['acquisition_index'], marker='o', linewidth=1.5, color='tab:red')
    axes[1].set_xlabel('date')
    axes[1].set_ylabel('acquisition index')
    axes[1].tick_params(axis='x', rotation=45)
else:
    axes[1].text(0.5, 0.5, 'No acquisition dates were found', ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_axis_off()
axes[1].set_title('Acquisition timeline')

plt.show()

## Useful CLI commands

The notebook API and the CLI are aligned. The same logical contracts shown above can be printed from the command line.

The two most useful commands are:

- inspect the Stage-1 contract and check one dataset
- print the logical contract for all stages

This is useful when you want a scriptable, non-notebook view of the same information.

In [ ]:
stage1_cli = 'rustamps describe-inputs --stage 1 --dataset <dataset> --patch PATCH_1'
all_stages_cli = 'rustamps describe-inputs --stage all'
stage1_dry_run = 'rustamps run --dataset <dataset> --start-step 1 --end-step 1 --dry-run'

print(stage1_cli)
print(all_stages_cli)
print(stage1_dry_run)

## What Stage 1 writes next

Once these inputs look sensible, Stage 1 writes structured outputs for the rest of the pipeline. The most important are:

- `ps1.mat`: organized geometry, timing, and indexing metadata
- `ph1.mat`: the complex phase stack reordered to match the Stage-1 metadata
- `bp1.mat`: baseline information aligned with the phase stack
- `da1.mat` and `hgt1.mat`: optional side products when those inputs exist

The operational habit is: inspect first, then run.

## Next step

Move to `02_rustamps_stage_execution.ipynb` once the Stage-1 inputs and checks make sense.

Return to `start_here.ipynb` if you need the broader map, or continue to `03_rustamps_verification.ipynb` after you have a run to compare.

## Proof: the inspection agrees with the files and with the Rust Stage 1

The tables above come from the retained Python inspector. Here its numbers are checked against independent file counts,
and against what the Rust `rustamps run` Stage 1 actually produces from the same inputs.

In [ ]:
import rustamps_nb as nb
patch_dir = DATASET / PATCH_NAME
overview = {r['metric']: r['value'] for r in SUMMARY['overview_rows']}
n_summary = int(overview['candidate count'])
n_ij = sum(1 for _ in open(patch_dir / 'pscands.1.ij'))
n_ll = (patch_dir / 'pscands.1.ll').stat().st_size // 8
n_da = sum(1 for _ in open(patch_dir / 'pscands.1.da'))
n_hgt = (patch_dir / 'pscands.1.hgt').stat().st_size // 4
n_ifg = (patch_dir / 'pscands.1.ph').stat().st_size // 8 // n_ij
statuses = {r['status'] for r in SUMMARY['consistency_rows']}

run = nb.stage_inputs('inspection')
nb.run_stage(run, 1)
n_rust = int(nb.read_mat(run / PATCH_NAME / 'ps1.mat')['n_ps'].ravel()[0])
n_rust_ifg = nb.read_mat(run / PATCH_NAME / 'ph1.mat')['ph'].shape[1]

proof = pd.DataFrame([
    nb.check(n_summary == n_ij == n_ll == n_da == n_hgt, 'the inspector counts the same candidates as the raw files', f'{n_summary} / {n_ij} / {n_ll} / {n_da} / {n_hgt}'),
    nb.check(statuses <= {'ok', 'derived at runtime'}, 'every consistency check is ok or derived at runtime', str(statuses)),
    nb.check(n_rust == n_summary, 'Rust Stage 1 keeps every candidate the inspector counted', f'{n_rust} == {n_summary}'),
    nb.check(n_rust_ifg == n_ifg + 1, 'ph1.mat has the raw interferograms plus the inserted null master', f'{n_rust_ifg} == {n_ifg} + 1'),
])
display(proof); assert proof.ok.all(); print('NOTEBOOK PASSED:', len(proof), 'checks')